# AROUSAL

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# === CONFIG ===
C_value=1
input_dir = 'features_d1_d2'  # path to feature files
target_type = 'Arousal'       # or 'Arousal'
output_csv = f'svm_{target_type.lower()}_c{C_value}_kernels_results.csv'
kernels = ['linear', 'poly', 'rbf', 'sigmoid']

# === Manual Label Mapping ===
def map_labels(df, target):
    if target == 'Valence':
        return df[target].map({'NV': 0, 'PV': 1})
    elif target == 'Arousal':
        return df[target].map({'LA': 0, 'HA': 1})
    else:
        raise ValueError("target_type must be 'Valence' or 'Arousal'")

# === RESULTS COLLECTOR ===
results = []

# === PROCESS EACH FILE ===
for filename in os.listdir(input_dir):
    if not filename.endswith('.csv'):
        continue

    channel = filename.split('_')[0]
    df = pd.read_csv(os.path.join(input_dir, filename))
    df = df.drop(columns=['Subject', 'Game','TotalPSD','TotalWaveletEnergy'])

    # Map labels
    df[target_type] = map_labels(df, target_type)

    # Split
    X = df.drop(columns=['Valence', 'Arousal']).values
    y = df[target_type].values

    # Scale
    X = StandardScaler().fit_transform(X)

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    # === Loop through kernels ===
    for kernel in kernels:
        all_preds = np.zeros_like(y)

        for train_idx, test_idx in skf.split(X, y):
            model = SVC(
                C=C_value,
                kernel=kernel,
                class_weight='balanced',
                max_iter=-1,  # no limit
                random_state=42
            )
            model.fit(X[train_idx], y[train_idx])
            all_preds[test_idx] = model.predict(X[test_idx])

        # Compute classwise totals and corrects
        class_0_total = np.sum(y == 0)
        class_1_total = np.sum(y == 1)
        class_0_correct = np.sum((y == 0) & (all_preds == 0))
        class_1_correct = np.sum((y == 1) & (all_preds == 1))

        results.append({
            'Channel': channel,
            'Kernel': kernel,
            'Class_0_Total': class_0_total,
            'Class_0_Correct': class_0_correct,
            'Class_1_Total': class_1_total,
            'Class_1_Correct': class_1_correct
        })

# === SAVE RESULTS ===
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)
print(f"\n✅ All SVM results saved to: {output_csv}")



✅ All SVM results saved to: svm_arousal_c100_kernels_results.csv


# VALENCE

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# === CONFIG ===
C_value=1
input_dir = 'features_d1_d2'  # path to feature files
target_type = 'Arousal'       # or 'Arousal'
output_csv = f'svm_{target_type.lower()}_c{C_value}_kernels_results.csv'
kernels = ['linear', 'poly', 'rbf', 'sigmoid']

# === Manual Label Mapping ===
def map_labels(df, target):
    if target == 'Valence':
        return df[target].map({'NV': 0, 'PV': 1})
    elif target == 'Arousal':
        return df[target].map({'LA': 0, 'HA': 1})
    else:
        raise ValueError("target_type must be 'Valence' or 'Arousal'")

# === RESULTS COLLECTOR ===
results = []

# === PROCESS EACH FILE ===
for filename in os.listdir(input_dir):
    if not filename.endswith('.csv'):
        continue

    channel = filename.split('_')[0]
    df = pd.read_csv(os.path.join(input_dir, filename))
    df = df.drop(columns=['Subject', 'Game','TotalPSD','TotalWaveletEnergy'])

    # Map labels
    df[target_type] = map_labels(df, target_type)

    # Split
    X = df.drop(columns=['Valence', 'Arousal']).values
    y = df[target_type].values

    # Scale
    X = StandardScaler().fit_transform(X)

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    # === Loop through kernels ===
    for kernel in kernels:
        all_preds = np.zeros_like(y)

        for train_idx, test_idx in skf.split(X, y):
            model = SVC(
                C=C_value,
                kernel=kernel,
                class_weight='balanced',
                max_iter=-1,  # no limit
                random_state=42
            )
            model.fit(X[train_idx], y[train_idx])
            all_preds[test_idx] = model.predict(X[test_idx])

        # Compute classwise totals and corrects
        class_0_total = np.sum(y == 0)
        class_1_total = np.sum(y == 1)
        class_0_correct = np.sum((y == 0) & (all_preds == 0))
        class_1_correct = np.sum((y == 1) & (all_preds == 1))

        results.append({
            'Channel': channel,
            'Kernel': kernel,
            'Class_0_Total': class_0_total,
            'Class_0_Correct': class_0_correct,
            'Class_1_Total': class_1_total,
            'Class_1_Correct': class_1_correct
        })

# === SAVE RESULTS ===
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)
print(f"\n✅ All SVM results saved to: {output_csv}")
